# Notebook 9b: LoRA Few-Shot

Trains a fresh LoRA adapter on 2,000 English plus 500 Spanish examples and
evaluates it on the same 1,000 Spanish test rows used for the zero-shot run.


Setup

In [1]:
!pip install -q transformers peft bitsandbytes accelerate trl
print("packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.9 MB/s eta 0:00:00
packages installed


In [2]:
import os, json, pickle, gc
import numpy as np, pandas as pd, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from sklearn.metrics import classification_report
import warnings; warnings.filterwarnings('ignore')

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("compute capability:", torch.cuda.get_device_capability())

GPU: Tesla T4
compute capability: (7, 5)


In [2]:
from google.colab import drive
drive.mount('/content/drive')

REPO= "Cyberbullying-Detection-in-Bilingual-English-Spanish-Text-A-Comparative-Study-"
if not os.path.exists(f'/content/drive/MyDrive/{REPO}'):
    !cd /content/drive/MyDrive/ && git clone https://github.com/YusrahS/{REPO}
%cd /content/drive/MyDrive/{REPO}

DATA_DIR = f'/content/drive/MyDrive/{REPO}/data'
D_OLD= '/content/drive/MyDrive/cyberbullying_results'
SHARED= f'/content/drive/MyDrive/{REPO}/bilingual results'
D= D_OLD; os.makedirs(D, exist_ok=True)
M= '/content/drive/MyDrive/cyberbullying_models';  os.makedirs(M, exist_ok=True)
CKPT= f'{M}/lora_few_shot_ckpt'

print("data :", DATA_DIR)
print("runs:", D)
print("shared:", SHARED)

Mounted at /content/drive
/content/drive/MyDrive/Cyberbullying-Detection-in-Bilingual-English-Spanish-Text-A-Comparative-Study-
data  : /content/drive/MyDrive/Cyberbullying-Detection-in-Bilingual-English-Spanish-Text-A-Comparative-Study-/data
runs  : /content/drive/MyDrive/cyberbullying_results
shared: /content/drive/MyDrive/Cyberbullying-Detection-in-Bilingual-English-Spanish-Text-A-Comparative-Study-/bilingual results


In [ ]:
with open(f'{DATA_DIR}/english_transformer_data.pkl','rb') as f: eng= pickle.load(f)
with open(f'{DATA_DIR}/spanish_transformer_data.pkl', 'rb') as f: spa =pickle.load(f)

english_train_texts,english_train_labels= eng['train_texts'], eng['train_labels']
spanish_train_texts,spanish_train_labels = spa['train_texts'], spa['train_labels']
spanish_test_texts,spanish_test_labels= spa['test_texts'],  spa['test_labels']

print("EN train:", len(english_train_texts), "| ES train:", len(spanish_train_texts),"| ES test:", len(spanish_test_texts))

EN train: 45899 | ES train: 19325 | ES test: 4141
split sizes match the corrected corpus


Prompt format

In [ ]:
def format_prompt(text, label=None):
    instruction= ("You are a content moderation assistant. "
                   "Classify the following social media post as either 'cyberbullying' or "
                   "'not cyberbullying'. Respond with only one word: cyberbullying or "
                   "not_cyberbullying.")
    prompt=f"[INST] {instruction}\n\nPost: {text} [/INST]"
    if label is not None:
        prompt += f" {'cyberbullying' if label== 1 else 'not_cyberbullying'}</s>"
    return prompt

def create_dataset(texts, labels):
    return Dataset.from_dict({"text": [format_prompt(t, l) for t, l in zip(texts, labels)],"label": list(labels)})
print(format_prompt("You are so stupid and nobody likes you", label=1))

[INST] You are a content moderation assistant. Classify the following social media post as either 'cyberbullying' or 'not cyberbullying'. Respond with only one word: cyberbullying or not_cyberbullying.

Post: You are so stupid and nobody likes you [/INST] cyberbullying</s>


Load the base model


In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.1"
cc= torch.cuda.get_device_capability()
DTYPE= torch.bfloat16 if cc[0] >= 8 else torch.float16
bnb_config= BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=DTYPE,
)
tokenizer= AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token =tokenizer.eos_token
tokenizer.padding_side= "right"

base_model= AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto", torch_dtype=DTYPE)
base_model.config.use_cache= False
base_model.config.pretraining_tp= 1
base_model.config.torch_dtype =DTYPE

for _, p in base_model.named_parameters():
    if p.dtype in (torch.bfloat16, torch.float32) and DTYPE == torch.float16:
        p.data = p.data.to(torch.float16)

dts={}
for _,p in base_model.named_parameters(): dts[p.dtype] = dts.get(p.dtype, 0)+ 1
print("parameter dtypes:",dts)
assert torch.bfloat16 not in dts or DTYPE == torch.bfloat16
print("base model ready")

using torch.float16


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

parameter dtypes: {torch.float16: 67, torch.uint8: 224}
base model ready


LoRA configuration

In [ ]:
lora_config= LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
print(lora_config)

LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'o_proj', 'k_proj', 'q_proj', 'v_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


Evaluation helpers


In [ ]:
TOK_POS= tokenizer.encode(" cyberbullying",add_special_tokens=False)[0]
TOK_NEG =tokenizer.encode("not_cyberbullying",add_special_tokens=False)[0]
print("label first-tokens:",TOK_NEG, TOK_POS, "| distinct:", TOK_POS != TOK_NEG)

@torch.no_grad()
def predict_lora(model, tokenizer, texts, batch_size=16):
    was_pad,was_cache = tokenizer.padding_side, model.config.use_cache
    tokenizer.padding_side= "left"
    model.config.use_cache = True
    model.eval()
    out=[]
    for i in range(0, len(texts), batch_size):
        enc= tokenizer([format_prompt(t) for t in texts[i:i+batch_size]],return_tensors="pt", padding=True, truncation=True,max_length=256).to(model.device)
        mask=enc['attention_mask']
        pos= mask.long().cumsum(-1) - 1
        pos.masked_fill_(mask == 0, 1)
        logits =model(input_ids=enc['input_ids'], attention_mask=mask,
                       position_ids=pos).logits[:, -1, :]
        pair =torch.stack([logits[:, TOK_NEG], logits[:, TOK_POS]], 1).float()
        out.append(torch.softmax(pair, 1).cpu().numpy())
        if (i // batch_size) % 20 == 0:
            print(f"  {min(i+batch_size, len(texts))}/{len(texts)}")
    tokenizer.padding_side,model.config.use_cache = was_pad, was_cache
    return np.concatenate(out)

def save_run(tag, probs, labels, texts):
    preds= probs.argmax(1)
    labels =np.asarray(labels)
    np.save(f'{D}/{tag}_preds.npy',preds)
    np.save(f'{D}/{tag}_probs.npy',probs.astype(np.float32))
    np.save(f'{D}/{tag}_labels.npy', labels)
    rep = classification_report(labels, preds, target_names=['Non-Abusive','Abusive'], output_dict=True, digits=4)
    json.dump(rep, open(f'{D}/{tag}_metrics.json','w'), indent=2)
    pd.DataFrame({'text': texts, 'true': labels, 'pred': preds, 'prob_abusive': probs[:,1]}).to_csv(f'{D}/{tag}_samples.csv', index=False)
    print(f"{tag:26s} n={len(preds):6d} macroF1 {rep['macro avg']['f1-score']:.4f} "
          f"NonAb {rep['Non-Abusive']['recall']:.4f} Ab {rep['Abusive']['recall']:.4f}")
    return rep

def stratified_sample(texts, labels, n=1000, seed=42):
    rng =np.random.RandomState(seed)
    t, l = np.array(texts), np.array(labels)
    i0 =rng.choice(np.where(l == 0)[0], n // 2, replace=False)
    i1= rng.choice(np.where(l == 1)[0], n // 2, replace=False)
    idx =np.concatenate([i0, i1]); rng.shuffle(idx)
    return t[idx].tolist(), l[idx].tolist()
print("helpers ready")

label first-tokens: 459 23449 | distinct: True
helpers ready


Train


In [ ]:
from accelerate.state import AcceleratorState, PartialState
AcceleratorState._reset_state(True); PartialState._reset_state()
MAX_TRAIN, FEW_SHOT_SIZE = 2000, 500
few_texts= list(english_train_texts[:MAX_TRAIN]) + list(spanish_train_texts[:FEW_SHOT_SIZE])
few_labels =list(english_train_labels[:MAX_TRAIN]) +list(spanish_train_labels[:FEW_SHOT_SIZE])
print("training samples:", len(few_texts), "| Spanish class balance:", np.bincount(spanish_train_labels[:FEW_SHOT_SIZE]))

few_ds=create_dataset(few_texts, few_labels)
base_model= prepare_model_for_kbit_training(base_model)
model_few=get_peft_model(base_model, lora_config)
for _, p in model_few.named_parameters():
    if p.requires_grad: p.data =p.data.to(torch.float32)
model_few.print_trainable_parameters()

args =SFTConfig(
    output_dir=CKPT,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate =2e-4,
    fp16=False, bf16=False,
    optim="paged_adamw_8bit",
    logging_steps=50,
    save_strategy="steps", save_steps=200, save_total_limit=2,
    warmup_steps=100, lr_scheduler_type="cosine",
    report_to="none", dataloader_num_workers=0,
    max_length= 256, packing =False, dataset_text_field="text", seed=42,
)

trainer= SFTTrainer(model=model_few, args=args, train_dataset=few_ds, processing_class=tokenizer)
resume = os.path.isdir(CKPT) and any(d.startswith('checkpoint-') for d in os.listdir(CKPT))
print("resuming from checkpoint" if resume else "starting fresh")
trainer.train(resume_from_checkpoint=resume)
model_few.save_pretrained(f'{M}/lora_few_shot_adapter')
tokenizer.save_pretrained(f'{M}/lora_few_shot_adapter')
print("adapter saved to", f'{M}/lora_few_shot_adapter')

training samples: 2500 | Spanish class balance: [253 247]
trainable params: 6,815,744 || all params: 7,248,547,840 || trainable%: 0.0940


Adding EOS to train dataset:   0%|          | 0/2500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2500 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2500 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2500 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


starting fresh


Step,Training Loss
50,2.310554
100,1.340638
150,1.270249
200,1.270650
250,1.231059
300,1.274046
350,1.184815
400,1.165538
450,1.238837
500,1.174851


adapter saved to /content/drive/MyDrive/cyberbullying_models/lora_few_shot_adapter


Evaluate

In [ ]:
es_t, es_l =stratified_sample(spanish_test_texts, spanish_test_labels)
probs= predict_lora(model_few, tokenizer, es_t)
rep_few=save_run('lora_en2es_fewshot', probs, es_l, es_t)

f =rep_few['macro avg']['f1-score']
ZS_TAGS = ['lora_en2es_zeroshot', 'lora_en2es_zeroshot_v2']
ZS_DIRS = [SHARED, D,'/content/drive/MyDrive/new_cyberbullying_results']
zs_path = next((f'{d}/{t}_metrics.json' for d in ZS_DIRS for t in ZS_TAGS
                if os.path.exists(f'{d}/{t}_metrics.json')), None)

if zs_path:
    z = json.load(open(zs_path))['macro avg']['f1-score']
    print(f"\nzero-shot {z:.4f} -> few-shot {f:.4f}"
          f"   ({100*(f-z):+.2f} pp from {FEW_SHOT_SIZE} Spanish examples)")
    print(f"zero-shot read from {os.path.dirname(zs_path)}")
else:
    print(f"\nzero-shot metrics not found on this account;"
          " compare manually against 0.4687")


  16/1000
  336/1000
  656/1000
  976/1000
lora_en2es_fewshot         n=  1000 macroF1 0.7910 NonAb 0.8000 Ab 0.7820

zero-shot metrics not on this account; compare manually against 0.4687
